In [3]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION WITH 8-LABEL RHETORICAL ROLES
# + FULL KG CONSTRUCTION
# + CHAIN-OF-THOUGHT (CoT) LLM-GUIDED SUMMARIZATION
#
# KEY ADDITIONS vs base code:
#
# ═══════════════════════════════════════════════════════════
# 🆕 FULL KG CONSTRUCTION
# ═══════════════════════════════════════════════════════════
# Builds a typed, role-aware legal knowledge graph with:
#   1. Entity nodes:  extracted by en_legal_ner_trf (loaded ONCE globally)
#   2. Relation edges: SVO triples from dependency parsing
#   3. Role-aware edges: argument→ruling, statute→ruling, etc.
#   4. Temporal chains: date/event ordering extracted from text
#   5. Path ranking: salient paths via importance × path_length scoring
#
# WHY KG HELPS ROUGE-L:
#   KG paths follow the logical flow of the judgment. When the decoder
#   attends to KG paths ordered as facts→issue→ruling, the generated
#   tokens follow the same order as the reference abstract — extending
#   LCS chains → higher ROUGE-L.
#
# ═══════════════════════════════════════════════════════════
# 🆕 CHAIN-OF-THOUGHT (CoT) GUIDED SUMMARIZATION
# ═══════════════════════════════════════════════════════════
# Uses CoT prompting to force structured summarization:
#   Step 1 — Extract legal issue
#   Step 2 — Extract court's reasoning
#   Step 3 — Extract the holding/decision
#   Step 4 — Synthesize into a coherent summary
#
# WHY CoT HELPS ROUGE-L:
#   Reference abstracts in In-Abs follow issue→reasoning→holding structure.
#   CoT prompting forces the model to generate in this exact order,
#   maximising sequential token overlap (LCS) with references.
#   This is the highest-leverage intervention for ROUGE-L improvement.
#
# OPTIMIZATIONS:
#   - spaCy (en_legal_ner_trf) loaded ONCE globally — not per sentence
#   - KG text representation rebuilt per document (not per sentence)
#   - CoT prompt uses KG-extracted entities for grounding
#   - All novel methods run in sequence: KG → CoT → LCS Fusion
# =========================================================

import os
import json
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration,
    LogitsProcessor,
    LogitsProcessorList
)
from collections import Counter, defaultdict
import evaluate
import nltk
from copy import deepcopy

# ── NLTK ──────────────────────────────────────────────────
print("📥 Downloading NLTK data...")
for resource in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource.split('/')[-1], quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# 🆕 LOAD spaCy ONCE GLOBALLY (en_legal_ner_trf)
# =========================================================
# CRITICAL OPTIMIZATION: spaCy is loaded exactly once here.
# All KG construction functions receive `nlp` as a parameter
# rather than calling spacy.load() each time.

print("\n📚 Loading spaCy legal NER model (en_legal_ner_trf)...")
try:
    import spacy
    nlp = spacy.load("en_legal_ner_trf")
    SPACY_AVAILABLE = True
    print("✅ en_legal_ner_trf loaded successfully")
except Exception as e:
    print(f"⚠️  en_legal_ner_trf unavailable ({e}), trying en_core_web_sm...")
    try:
        nlp = spacy.load("en_core_web_sm")
        SPACY_AVAILABLE = True
        print("✅ en_core_web_sm loaded as fallback")
    except Exception as e2:
        print(f"⚠️  No spaCy model available ({e2}). Using regex fallback.")
        nlp = None
        SPACY_AVAILABLE = False

# =========================================================
# CONFIGURATION
# =========================================================

MODEL_PATHS = {
    "BART":       "BART",
    "LED":        "LED",
    "PEGASUS":    "PEGASUS",
    "ROLE_MODEL": "best_RR_model"
}

TRAIN_JSON_PATH = "train.json"
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./ensemble_results_fullkg_cot"

ROLE_CLASSIFICATION_FILE = "sentence_role_annotations_8label.json"
ROLE_CONTRIBUTION_FILE   = "role_contribution_scores_8label.json"
ROLE_WEIGHTS_FILE        = "normalized_role_weights_8label.json"

MODEL_CONFIG = {
    "BART":    {"max_input": 1024, "max_output": 256, "num_beams": 4},
    "LED":     {"max_input": 4096, "max_output": 512, "num_beams": 4},
    "PEGASUS": {"max_input": 1024, "max_output": 256, "num_beams": 4}
}

HSLN_LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE"
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE", "FACTS", "ISSUE", "ARGUMENTS",
    "REASONING", "PRECEDENT_RELIED", "PRECEDENT_NOT_RELIED", "PROCEDURAL"
]
NUM_LABELS = 8

LABEL_MAPPING_13_TO_8 = {
    "PREAMBLE":       "PREAMBLE",  "ISSUE":          "ISSUE",
    "FAC":            "FACTS",     "PRE_RELIED":     "PRECEDENT_RELIED",
    "PRE_NOT_RELIED": "PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER": "ARGUMENTS", "ARG_RESPONDENT": "ARGUMENTS",
    "ANALYSIS":       "REASONING", "RATIO":          "REASONING",
    "RPC":            "REASONING", "STA":            "PROCEDURAL",
    "RLC":            "PROCEDURAL","NONE":           "PROCEDURAL"
}

def map_13_to_8_labels(labels_13):
    return [LABEL_MAPPING_13_TO_8.get(l, "PROCEDURAL") for l in labels_13]

HYBRID_ALPHA = 0.6
HYBRID_BETA  = 0.4

COMPRESSION_RATIOS = {"BART": 0.12, "PEGASUS": 0.12, "LED": 0.40}
MIN_SENTENCES      = {"BART": 30,   "PEGASUS": 30,   "LED": 100}
MAX_SENTENCES      = {"BART": 150,  "PEGASUS": 150,  "LED": 400}

def get_adaptive_targets(doc_length):
    t = {}
    for m, r in COMPRESSION_RATIOS.items():
        v = int(doc_length * r)
        t[m] = max(MIN_SENTENCES[m], min(MAX_SENTENCES[m], v))
    return t

MAX_TRAIN_SAMPLES = 3000
ENSEMBLE_WEIGHTS  = {"BART": 0.25, "LED": 0.50, "PEGASUS": 0.25}

# KG-DIFF CONFIG
KG_CRITICAL_ROLES       = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]
KG_SEMANTIC_THRESHOLD   = 0.75
KG_COVERAGE_THRESHOLD   = 0.75
KG_MAX_ITERATIONS       = 2
KG_TOP_MISSING          = 5
KG_MAX_CORRECTION_SENTS = 10

# LCS CONFIG
LCS_ANCHOR_ROLES           = ["ISSUE", "REASONING", "PRECEDENT_RELIED", "FACTS"]
LCS_MIN_NGRAM_OVERLAP      = 3
LCS_CONNECTIVES            = [
    "The court held that", "It was observed that", "Therefore,",
    "Furthermore,", "The appellant contended that",
    "In view of the above,", "The High Court noted that", "Accordingly,"
]
LCS_MAX_OUTPUT_SENTENCES   = 20
LCS_ANCHOR_LCS_WEIGHT      = 0.5
LCS_ANCHOR_SEMANTIC_WEIGHT = 0.5

# =========================================================
# 🆕 FULL KG CONSTRUCTION CONFIGURATION
# =========================================================

# Role → node type mapping for the knowledge graph
ROLE_TO_NODE_TYPE = {
    "PREAMBLE":           "CASE_INFO",
    "FACTS":              "FACT",
    "ISSUE":              "LEGAL_ISSUE",
    "ARGUMENTS":          "ARGUMENT",
    "REASONING":          "RULING",
    "PRECEDENT_RELIED":   "STATUTE_PRECEDENT",
    "PRECEDENT_NOT_RELIED":"STATUTE_PRECEDENT",
    "PROCEDURAL":         "PROCEDURAL"
}

# Role-aware edge types (source_role → target_role → edge_label)
# These encode the logical flow of Indian legal judgments
ROLE_EDGE_RULES = {
    ("FACTS",     "LEGAL_ISSUE"):        "GIVES_RISE_TO",
    ("ARGUMENT",  "RULING"):             "ADDRESSED_BY",
    ("LEGAL_ISSUE","RULING"):            "RESOLVED_BY",
    ("STATUTE_PRECEDENT", "RULING"):     "SUPPORTS",
    ("RULING",    "PROCEDURAL"):         "RESULTS_IN",
    ("FACT",      "ARGUMENT"):           "SUPPORTS",
    ("CASE_INFO", "LEGAL_ISSUE"):        "CONCERNS",
}

# Entity types to extract as KG nodes (from en_legal_ner_trf or en_core_web_sm)
LEGAL_ENTITY_TYPES = {
    "COURT", "JUDGE", "LAWYER", "PETITIONER", "RESPONDENT",
    "STATUTE", "PROVISION", "PRECEDENT", "DATE", "GPE",
    "ORG", "PERSON", "LAW", "CARDINAL", "ORDINAL"
}

# Path ranking: max path length to explore in the KG
KG_MAX_PATH_LENGTH = 4

# Top-K salient KG paths to include in CoT prompt grounding
KG_TOP_K_PATHS = 10

# =========================================================
# 🆕 CHAIN-OF-THOUGHT PROMPTING CONFIGURATION
# =========================================================

# CoT prompt template for legal summarization.
# {issue}    = KG-extracted legal issue sentences
# {reasoning}= KG-extracted reasoning sentences
# {decision} = KG-extracted decision sentences
# {entities} = top KG entities (parties, statutes, courts)
COT_PROMPT_TEMPLATE = """You are a legal summarization expert. Summarize the following legal judgment by following these steps carefully:

STEP 1 - LEGAL ISSUE: Identify the core legal question being decided.
Key issue sentences: {issue}

STEP 2 - COURT'S REASONING: Explain the court's reasoning and analysis.
Key reasoning sentences: {reasoning}

STEP 3 - HOLDING/DECISION: State the court's final decision.
Key decision sentences: {decision}

STEP 4 - SYNTHESIS: Write a coherent, concise legal summary in 3-5 sentences that covers the issue, reasoning, and holding in that order.
Important entities: {entities}

Full document context:
{context}

Structured Summary (issue → reasoning → holding):"""

# Shorter CoT prompt for models with limited input window (BART/PEGASUS)
COT_PROMPT_SHORT = """Summarize this legal judgment following: issue → reasoning → holding.

Issue: {issue}
Reasoning: {reasoning}
Decision: {decision}
Entities: {entities}

Summary (3-5 sentences, issue first, then reasoning, then holding):"""

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\n" + "="*70)
print("📋 CONFIGURATION — FULL KG + CoT SUMMARIZATION")
print("="*70)
print("  🆕 Full KG Construction with en_legal_ner_trf (loaded once)")
print("  🆕 Chain-of-Thought (CoT) Guided Summarization")
print("  ✨ LCS-Guided Sentence Fusion (ROUGE-L booster)")
print(f"  Output: {OUTPUT_DIR}")
print("="*70)


# =========================================================
# HSLN MODEL DEFINITION (unchanged)
# =========================================================

class PrototypeAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h = heads; self.dh = dim // heads
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
        self.o = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape; C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, dim=-1); K = F.normalize(K, dim=-1)
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            outs.append(self.dropout(a) @ V[:, h])
        return self.ln(x + self.dropout(self.o(torch.cat(outs, -1))))


class RPL(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))

    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T


class RTM(nn.Module):
    def __init__(self, num_labels, rtm_lambda):
        super().__init__()
        self.A = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.rtm_lambda = rtm_lambda

    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1)
            logits[:, t] += self.rtm_lambda * tr
        return logits


class HSLNModel(nn.Module):
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.lstm_hidden = lstm_hidden
        self.num_labels = num_labels
        self.att  = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, 2, bidirectional=True,
                            batch_first=True, dropout=dropout)
        self.cls  = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl  = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm  = RTM(num_labels, rtm_lambda)
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))

    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(),
                                                     batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        a = torch.sigmoid(self.alpha)
        return self.rtm(a * self.cls(o) + (1 - a) * self.rpl(o))

    def predict(self, embeddings, lengths=None):
        return torch.argmax(self.forward(embeddings, lengths), dim=-1)


# ── Load HSLN ─────────────────────────────────────────────
print("\n📚 Loading HSLN role classifier...")
config_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(config_path):
    with open(config_path) as f: cfg = json.load(f)
    embedding_dim = cfg.get('embedding_dim', 768)
    lstm_hidden   = cfg.get('lstm_hidden', 384)
    dropout       = cfg.get('dropout', 0.3)
    rtm_lambda    = cfg.get('rtm_lambda', 0.05)
else:
    embedding_dim, lstm_hidden, dropout, rtm_lambda = 768, 384, 0.3, 0.05

role_model = HSLNModel(embedding_dim, lstm_hidden, NUM_HSLN_LABELS, dropout, rtm_lambda)
state_dict = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"], "pytorch_model.bin"),
                        map_location=device)
state_dict.pop('prototypes', None)
role_model.load_state_dict(state_dict, strict=False)
role_model.prototypes = torch.load(
    os.path.join(MODEL_PATHS["ROLE_MODEL"], "prototypes.pt"), map_location=device)
role_model.to(device).eval()
print("✅ HSLN loaded")

print("\n📚 Loading InLegalBERT...")
legal_model_name = "law-ai/InLegalBERT"
legal_tokenizer  = AutoTokenizer.from_pretrained(legal_model_name)
legal_model      = AutoModel.from_pretrained(legal_model_name).to(device)
legal_model.eval()
print("✅ InLegalBERT loaded")


# =========================================================
# EMBEDDING & ROLE CLASSIFICATION
# =========================================================

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    if not texts: return np.array([])
    embs = []
    for i in range(0, len(texts), batch_size):
        b   = texts[i:i+batch_size]
        enc = legal_tokenizer(b, padding=True, truncation=True,
                              max_length=512, return_tensors="pt").to(device)
        out  = legal_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        embs.append(((out * mask).sum(1) / mask.sum(1)).cpu().numpy())
    return np.vstack(embs)


@torch.no_grad()
def classify_roles(sentences, batch_size=16):
    if not sentences: return []
    preds = []
    for i in range(0, len(sentences), batch_size):
        b      = sentences[i:i+batch_size]
        inputs = legal_tokenizer(b, padding=True, truncation=True,
                                 max_length=128, return_tensors="pt").to(device)
        emb    = legal_model(**inputs).last_hidden_state.mean(dim=1).unsqueeze(1)
        lens   = torch.ones(len(b), dtype=torch.long).to(device)
        preds.extend(role_model.predict(emb, lens)[:, 0].cpu().tolist())
    return map_13_to_8_labels([HSLN_LABELS[p] for p in preds])


# =========================================================
# ═══════════════════════════════════════════════════════════
# 🆕 FULL KNOWLEDGE GRAPH CONSTRUCTION
# ═══════════════════════════════════════════════════════════
# =========================================================

class LegalKnowledgeGraph:
    """
    Full legal knowledge graph for a single document.

    Nodes:
        Each node has: id, text, node_type (from ROLE_TO_NODE_TYPE),
        role, sent_idx, entities (list of named entities from spaCy),
        importance (float), embedding (np.ndarray)

    Edges:
        Each edge has: source_id, target_id, edge_type, weight

    The graph encodes:
        1. Entity co-occurrence edges (two sentences sharing an entity are connected)
        2. Role-flow edges (facts→issue, argument→ruling, etc.)
        3. Temporal chain edges (date-ordered sentences are connected)
        4. SVO-triple edges (subject→verb→object extracted by dependency parsing)
    """

    def __init__(self):
        self.nodes = {}          # node_id → dict
        self.edges = []          # list of edge dicts
        self.entity_index = defaultdict(list)   # entity_text → list of node_ids
        self._next_id = 0

    def add_node(self, text, node_type, role, sent_idx,
                 entities=None, importance=0.0, embedding=None):
        nid = self._next_id
        self._next_id += 1
        self.nodes[nid] = {
            "id":         nid,
            "text":       text,
            "node_type":  node_type,
            "role":       role,
            "sent_idx":   sent_idx,
            "entities":   entities or [],
            "importance": importance,
            "embedding":  embedding
        }
        for ent in (entities or []):
            self.entity_index[ent.lower()].append(nid)
        return nid

    def add_edge(self, source_id, target_id, edge_type, weight=1.0):
        if source_id != target_id:
            self.edges.append({
                "source":    source_id,
                "target":    target_id,
                "edge_type": edge_type,
                "weight":    weight
            })

    def get_neighbors(self, node_id, edge_types=None):
        """Return list of (neighbor_id, edge_type, weight) for a given node."""
        neighbors = []
        for e in self.edges:
            if e["source"] == node_id:
                if edge_types is None or e["edge_type"] in edge_types:
                    neighbors.append((e["target"], e["edge_type"], e["weight"]))
            elif e["target"] == node_id:
                if edge_types is None or e["edge_type"] in edge_types:
                    neighbors.append((e["source"], e["edge_type"] + "_REV", e["weight"]))
        return neighbors

    def to_summary(self):
        """Return a compact string summary of the graph for diagnostics."""
        return (f"LegalKG: {len(self.nodes)} nodes, {len(self.edges)} edges, "
                f"{len(self.entity_index)} unique entities")


def extract_entities_from_text(text, max_chars=512):
    """
    Extract named entities from text using the globally loaded spaCy model.
    Uses the GLOBAL `nlp` object — NOT reloaded.

    Args:
        text:      str — sentence or paragraph text
        max_chars: int — truncate to avoid OOM on very long sentences

    Returns:
        entities: List[str] — unique entity texts relevant to legal domain
    """
    if not SPACY_AVAILABLE or nlp is None:
        return _extract_entities_regex(text)

    doc      = nlp(text[:max_chars])
    entities = []
    for ent in doc.ents:
        if ent.label_ in LEGAL_ENTITY_TYPES and len(ent.text.strip()) > 1:
            entities.append(ent.text.strip())
    return list(set(entities))


def _extract_entities_regex(text):
    """Fallback entity extraction using regex (when spaCy unavailable)."""
    # Capitalised phrases as proxy for named entities
    matches = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    return list(set(m for m in matches if len(m) > 3))


def extract_svo_triples_from_doc(sentences):
    """
    Extract (subject, verb, object) triples for all sentences at once
    using the globally loaded spaCy model.

    OPTIMIZATION: processes all sentences in ONE spaCy pipeline pass
    (using nlp.pipe), not one call per sentence.

    Args:
        sentences: List[str]

    Returns:
        triples_per_sent: List[List[Tuple]] — one list of triples per sentence
    """
    if not SPACY_AVAILABLE or nlp is None:
        return [_extract_triples_fallback_single(s) for s in sentences]

    triples_per_sent = []
    # nlp.pipe batches sentences efficiently
    for doc in nlp.pipe(sentences, batch_size=32):
        triples = []
        for token in doc:
            if token.dep_ == "ROOT" and token.pos_ == "VERB":
                subjs = [c for c in token.children if c.dep_ in ("nsubj", "nsubjpass")]
                objs  = [c for c in token.children if c.dep_ in ("dobj", "pobj", "attr", "oprd")]
                for s in subjs:
                    for o in objs:
                        st = " ".join(t.text for t in s.subtree if not t.is_stop).lower().strip()
                        ot = " ".join(t.text for t in o.subtree if not t.is_stop).lower().strip()
                        rt = token.lemma_.lower()
                        if st and ot and rt:
                            triples.append((st, rt, ot))
        triples_per_sent.append(triples)
    return triples_per_sent


def _extract_triples_fallback_single(sentence):
    words = re.findall(r'\b[A-Za-z][a-z]+(?:\s+[A-Z][a-z]+)*\b', sentence)
    words = [w.lower() for w in words if len(w) > 3]
    return [(words[i], "related_to", words[i+1]) for i in range(len(words)-1)]


def extract_temporal_order(sentences):
    """
    Extract a rough temporal ordering of sentences based on date mentions.
    Returns list of (sent_idx, date_mention) sorted by position in text.

    Used to add temporal-chain edges in the KG.
    """
    date_pattern = re.compile(
        r'\b(\d{1,2}[-/]\d{1,2}[-/]\d{2,4}|\d{4}|'
        r'January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\b', re.IGNORECASE
    )
    ordered = []
    for idx, sent in enumerate(sentences):
        if date_pattern.search(sent):
            ordered.append(idx)
    return ordered


def build_full_legal_kg(doc_annotation, normalized_weights):
    """
    🆕 Build a full typed legal knowledge graph for a document.

    Pipeline:
      1. Create one node per source sentence (with role, node_type)
      2. Extract all entities via en_legal_ner_trf (one nlp.pipe call)
      3. Extract SVO triples via dependency parsing (one nlp.pipe call)
      4. Compute node importance = role_weight × semantic centroid similarity
      5. Add entity co-occurrence edges
      6. Add role-flow edges (facts→issue, argument→ruling, etc.)
      7. Add temporal chain edges
      8. Add SVO triple edges (subject→object via verb)

    Args:
        doc_annotation:     dict — sentence annotations with roles
        normalized_weights: dict — role → weight

    Returns:
        kg:      LegalKnowledgeGraph
        kg_log:  dict — construction diagnostics
    """
    kg  = LegalKnowledgeGraph()
    log = {"method": "full_legal_kg_construction", "steps": []}

    sentences  = [s["sentence"] for s in doc_annotation["sentences"]]
    roles      = [s["role"]     for s in doc_annotation["sentences"]]
    sent_indices = [s["index"]  for s in doc_annotation["sentences"]]

    if not sentences:
        log["early_exit"] = "No sentences"
        return kg, log

    # ── STEP 1: Embed all sentences (one batch call) ──────────────────────
    embeddings   = embed_with_legalbert(sentences)
    centroid_emb = embeddings.mean(axis=0)

    # ── STEP 2: Extract entities for ALL sentences (one nlp.pipe call) ────
    # OPTIMIZATION: single nlp.pipe call processes all sentences together
    all_entities_per_sent = []
    if SPACY_AVAILABLE and nlp is not None:
        for doc in nlp.pipe(sentences, batch_size=32):
            ents = [e.text.strip() for e in doc.ents
                    if e.label_ in LEGAL_ENTITY_TYPES and len(e.text.strip()) > 1]
            all_entities_per_sent.append(list(set(ents)))
    else:
        all_entities_per_sent = [_extract_entities_regex(s) for s in sentences]

    log["steps"].append({
        "step": 2, "name": "Entity extraction",
        "total_entities": sum(len(e) for e in all_entities_per_sent),
        "method": "en_legal_ner_trf (single nlp.pipe pass)"
    })

    # ── STEP 3: Extract SVO triples (one nlp.pipe call) ────────────────────
    svo_per_sent = extract_svo_triples_from_doc(sentences)

    log["steps"].append({
        "step": 3, "name": "SVO triple extraction",
        "total_triples": sum(len(t) for t in svo_per_sent),
        "method": "dependency_parsing (single nlp.pipe pass)"
    })

    # ── STEP 4: Create nodes with importance scores ────────────────────────
    node_ids = []
    for i, (sent, role, sidx) in enumerate(zip(sentences, roles, sent_indices)):
        cos_sim    = float(cosine_similarity([embeddings[i]], [centroid_emb])[0][0])
        role_w     = normalized_weights.get(role, 0.1)
        importance = role_w * cos_sim
        node_type  = ROLE_TO_NODE_TYPE.get(role, "PROCEDURAL")

        nid = kg.add_node(
            text       = sent,
            node_type  = node_type,
            role       = role,
            sent_idx   = sidx,
            entities   = all_entities_per_sent[i],
            importance = importance,
            embedding  = embeddings[i]
        )
        node_ids.append(nid)

    log["steps"].append({
        "step": 4, "name": "Node creation",
        "total_nodes": len(node_ids),
        "avg_importance": round(np.mean([kg.nodes[n]["importance"] for n in node_ids]), 4)
    })

    # ── STEP 5: Entity co-occurrence edges ─────────────────────────────────
    # Two nodes share an edge if they mention the same named entity
    entity_edges_added = 0
    for entity, nid_list in kg.entity_index.items():
        if len(nid_list) < 2: continue
        for i in range(len(nid_list)):
            for j in range(i+1, min(i+5, len(nid_list))):   # limit fan-out
                ni, nj = nid_list[i], nid_list[j]
                # Weight by entity importance: longer entity = more specific = higher weight
                w = min(1.0, len(entity.split()) / 3.0)
                kg.add_edge(ni, nj, "ENTITY_CO_OCCURRENCE", weight=w)
                entity_edges_added += 1

    log["steps"].append({
        "step": 5, "name": "Entity co-occurrence edges",
        "edges_added": entity_edges_added
    })

    # ── STEP 6: Role-flow edges ─────────────────────────────────────────────
    # Connect nodes based on the logical flow rules in ROLE_EDGE_RULES
    role_edges_added = 0
    role_to_node_ids = defaultdict(list)
    for nid, node in kg.nodes.items():
        role_to_node_ids[node["node_type"]].append(nid)

    for (src_type, tgt_type), edge_label in ROLE_EDGE_RULES.items():
        src_nodes = role_to_node_ids.get(src_type, [])
        tgt_nodes = role_to_node_ids.get(tgt_type, [])
        if not src_nodes or not tgt_nodes: continue

        # Connect the top-3 most important source nodes to top-3 target nodes
        top_srcs = sorted(src_nodes, key=lambda n: kg.nodes[n]["importance"], reverse=True)[:3]
        top_tgts = sorted(tgt_nodes, key=lambda n: kg.nodes[n]["importance"], reverse=True)[:3]

        for si in top_srcs:
            for ti in top_tgts:
                # Use semantic similarity as edge weight
                si_emb = kg.nodes[si]["embedding"]
                ti_emb = kg.nodes[ti]["embedding"]
                if si_emb is not None and ti_emb is not None:
                    w = float(cosine_similarity([si_emb], [ti_emb])[0][0])
                else:
                    w = 0.5
                kg.add_edge(si, ti, edge_label, weight=w)
                role_edges_added += 1

    log["steps"].append({
        "step": 6, "name": "Role-flow edges",
        "edges_added": role_edges_added
    })

    # ── STEP 7: Temporal chain edges ────────────────────────────────────────
    temporal_indices = extract_temporal_order(sentences)
    temporal_edges   = 0
    for i in range(len(temporal_indices) - 1):
        ni = node_ids[temporal_indices[i]]
        nj = node_ids[temporal_indices[i+1]]
        kg.add_edge(ni, nj, "TEMPORAL_PRECEDES", weight=1.0)
        temporal_edges += 1

    log["steps"].append({
        "step": 7, "name": "Temporal chain edges",
        "edges_added": temporal_edges
    })

    # ── STEP 8: SVO triple edges ────────────────────────────────────────────
    svo_edges = 0
    for i, triples in enumerate(svo_per_sent):
        ni = node_ids[i]
        for (subj, verb, obj) in triples:
            # Find nodes that mention the object entity
            obj_tokens = obj.split()
            for candidate_id in node_ids:
                if candidate_id == ni: continue
                cand_text = kg.nodes[candidate_id]["text"].lower()
                if any(tok in cand_text for tok in obj_tokens if len(tok) > 3):
                    kg.add_edge(ni, candidate_id, f"SVO_{verb.upper()}", weight=0.6)
                    svo_edges += 1
                    break   # one SVO edge per triple

    log["steps"].append({
        "step": 8, "name": "SVO triple edges",
        "edges_added": svo_edges
    })

    log["total_edges"]    = len(kg.edges)
    log["total_nodes"]    = len(kg.nodes)
    log["kg_summary"]     = kg.to_summary()
    return kg, log


def rank_kg_paths(kg, normalized_weights, top_k=KG_TOP_K_PATHS,
                  max_path_length=KG_MAX_PATH_LENGTH):
    """
    Find and rank the top-K salient KG paths.

    Path scoring:
        path_score = mean(node.importance) × path_length × role_coverage_bonus
        role_coverage_bonus: +0.2 per unique critical role (ISSUE, REASONING, PRECEDENT)
        covered in the path

    Algorithm: Greedy DFS from the highest-importance node (ISSUE/RULING type),
               limited to max_path_length hops.

    Args:
        kg:                 LegalKnowledgeGraph
        normalized_weights: dict — role → weight
        top_k:              int  — number of paths to return
        max_path_length:    int  — max path length

    Returns:
        ranked_paths: List[List[int]] — each path is a list of node_ids
        path_texts:   List[str]       — text representation of each path
    """
    if not kg.nodes:
        return [], []

    # Start DFS from all LEGAL_ISSUE and RULING nodes (highest value sources)
    start_node_types = {"LEGAL_ISSUE", "RULING"}
    start_nodes = sorted(
        [nid for nid, node in kg.nodes.items()
         if node["node_type"] in start_node_types],
        key=lambda n: kg.nodes[n]["importance"],
        reverse=True
    )

    if not start_nodes:
        # Fallback: start from highest-importance nodes of any type
        start_nodes = sorted(kg.nodes.keys(),
                             key=lambda n: kg.nodes[n]["importance"],
                             reverse=True)[:5]

    all_paths  = []
    seen_paths = set()

    def dfs(current, path, visited):
        if len(path) > max_path_length:
            return
        path_key = tuple(path)
        if path_key in seen_paths:
            return
        if len(path) >= 2:
            all_paths.append(list(path))
            seen_paths.add(path_key)
        neighbors = kg.get_neighbors(current)
        for (nbr, etype, w) in sorted(neighbors, key=lambda x: x[2], reverse=True)[:3]:
            if nbr not in visited:
                visited.add(nbr)
                dfs(nbr, path + [nbr], visited)
                visited.remove(nbr)

    for start in start_nodes[:10]:
        dfs(start, [start], {start})

    # Score all found paths
    scored_paths = []
    for path in all_paths:
        nodes_in_path = [kg.nodes[n] for n in path]
        mean_imp      = np.mean([n["importance"] for n in nodes_in_path])
        roles_covered = set(n["role"] for n in nodes_in_path)
        crit_coverage = len(roles_covered & set(KG_CRITICAL_ROLES))
        path_score    = mean_imp * len(path) * (1.0 + 0.2 * crit_coverage)
        scored_paths.append((path_score, path))

    scored_paths.sort(key=lambda x: x[0], reverse=True)
    top_paths = [p for _, p in scored_paths[:top_k]]

    # Convert to text
    path_texts = []
    for path in top_paths:
        sents     = [kg.nodes[n]["text"][:100] for n in path]
        edge_types = []
        for i in range(len(path) - 1):
            for e in kg.edges:
                if e["source"] == path[i] and e["target"] == path[i+1]:
                    edge_types.append(e["edge_type"])
                    break
            else:
                edge_types.append("→")
        parts = [sents[0]]
        for i, et in enumerate(edge_types):
            parts.append(f"  [{et}]  ")
            parts.append(sents[i+1])
        path_texts.append("".join(parts))

    return top_paths, path_texts


def extract_kg_context_for_cot(kg, doc_annotation, normalized_weights):
    """
    Extract structured context from the KG for Chain-of-Thought prompting.

    Returns:
        issue_text:    str — top ISSUE sentences (joined)
        reasoning_text:str — top REASONING sentences (joined)
        decision_text: str — top PROCEDURAL/RULING sentences (joined)
        entities_text: str — top entities across the document
        salient_paths: str — top KG path descriptions
    """
    # Collect top sentences per role by node importance
    role_to_sents = defaultdict(list)
    for nid, node in kg.nodes.items():
        role_to_sents[node["role"]].append((node["importance"], node["text"]))

    def top_sents(role, n=2):
        sents = role_to_sents.get(role, [])
        sents.sort(reverse=True)
        return " ".join(s[:200] for _, s in sents[:n])

    issue_text    = top_sents("ISSUE",     n=2)
    reasoning_text= top_sents("REASONING", n=3)
    decision_text = (top_sents("PROCEDURAL", n=1) + " " +
                     top_sents("PRECEDENT_RELIED", n=1)).strip()

    # Top entities: most frequent across high-importance nodes
    entity_freq = Counter()
    for nid, node in kg.nodes.items():
        if node["importance"] > 0.3:
            for ent in node["entities"]:
                entity_freq[ent] += 1
    top_entities  = [e for e, _ in entity_freq.most_common(8)]
    entities_text = ", ".join(top_entities) if top_entities else "N/A"

    # Top KG paths
    _, path_texts = rank_kg_paths(kg, normalized_weights, top_k=3)
    salient_paths = "; ".join(path_texts[:3]) if path_texts else ""

    return issue_text, reasoning_text, decision_text, entities_text, salient_paths


# =========================================================
# ═══════════════════════════════════════════════════════════
# 🆕 CHAIN-OF-THOUGHT (CoT) GUIDED SUMMARIZATION
# ═══════════════════════════════════════════════════════════
# =========================================================

def build_cot_prompt(issue_text, reasoning_text, decision_text,
                      entities_text, context, model_name="LED"):
    """
    Build the Chain-of-Thought prompt for legal summarization.

    For LED (large input window) → use the full COT_PROMPT_TEMPLATE.
    For BART/PEGASUS (smaller window) → use COT_PROMPT_SHORT.

    The prompt enforces issue → reasoning → holding order, which directly
    matches the structure of reference abstracts in In-Abs, maximising
    the LCS between generated and reference summaries.

    Args:
        issue_text:    str — KG-extracted issue sentences
        reasoning_text:str — KG-extracted reasoning sentences
        decision_text: str — KG-extracted decision sentences
        entities_text: str — top entities from KG
        context:       str — filtered source text
        model_name:    str — model to use (determines prompt length)

    Returns:
        prompt: str — full CoT prompt
    """
    # Truncate components to fit within token budget
    if model_name == "LED":
        # LED can handle 4096 tokens — use full template with longer context
        ctx_limit    = 2000
        issue_limit  = 300
        reason_limit = 400
        dec_limit    = 200
        template     = COT_PROMPT_TEMPLATE
    else:
        # BART/PEGASUS — shorter prompt
        ctx_limit    = 500
        issue_limit  = 150
        reason_limit = 200
        dec_limit    = 150
        template     = COT_PROMPT_SHORT

    issue_t    = (issue_text    or "Not explicitly stated")[:issue_limit]
    reason_t   = (reasoning_text or "See context")[:reason_limit]
    dec_t      = (decision_text  or "See context")[:dec_limit]
    context_t  = context[:ctx_limit]

    return template.format(
        issue     = issue_t,
        reasoning = reason_t,
        decision  = dec_t,
        entities  = entities_text,
        context   = context_t
    )


def generate_cot_summary(model_name, filtered_text, doc_annotation,
                          kg, normalized_weights):
    """
    🆕 Generate summary using Chain-of-Thought guided prompting.

    Workflow:
      1. Extract issue / reasoning / decision context from KG
      2. Build CoT prompt with extracted context
      3. Generate summary with the model using the CoT prompt
      4. Post-process: ensure the summary starts with the issue

    Args:
        model_name:         str — "BART", "LED", or "PEGASUS"
        filtered_text:      str — hybrid-selected source text
        doc_annotation:     dict — sentence-role annotations
        kg:                 LegalKnowledgeGraph
        normalized_weights: dict — role → weight

    Returns:
        summary: str  — CoT-guided summary
        cot_log: dict — diagnostics
    """
    issue_text, reasoning_text, decision_text, entities_text, paths = \
        extract_kg_context_for_cot(kg, doc_annotation, normalized_weights)

    cot_log = {
        "method":          "chain_of_thought_guided_summarization",
        "model":           model_name,
        "issue_chars":     len(issue_text),
        "reasoning_chars": len(reasoning_text),
        "decision_chars":  len(decision_text),
        "entities":        entities_text,
        "kg_paths_used":   len(paths.split(";")) if paths else 0
    }

    # Build CoT prompt
    prompt = build_cot_prompt(
        issue_text, reasoning_text, decision_text,
        entities_text, filtered_text, model_name
    )
    cot_log["prompt_chars"] = len(prompt)

    # Generate with standard model
    model     = models[model_name]
    tokenizer = tokenizers[model_name]
    config    = MODEL_CONFIG[model_name]

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=config["max_input"]
    ).to(device)

    with torch.no_grad():
        try:
            if model_name == "LED":
                gam = torch.zeros_like(inputs["attention_mask"])
                gam[:, 0] = 1
                ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    global_attention_mask=gam,
                    max_length=config["max_output"],
                    num_beams=config["num_beams"],
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
            else:
                ids = model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    max_length=config["max_output"],
                    num_beams=config["num_beams"],
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
            summary = tokenizer.decode(ids[0], skip_special_tokens=True)
        except Exception as e:
            # Fallback to standard summary if CoT prompt fails
            print(f"    ⚠️  CoT generation failed ({e}), falling back.")
            summary = generate_summary(model_name, filtered_text)
            cot_log["fallback"] = str(e)

    # ── Post-process: strip any leaked prompt text ─────────────────────────
    # Models sometimes echo the "Summary:" prefix — remove it
    for prefix in ["Structured Summary", "Summary (", "Summary:", "STEP 4"]:
        if prefix.lower() in summary.lower():
            idx = summary.lower().find(prefix.lower())
            candidate = summary[idx + len(prefix):].strip(" :\n")
            if len(candidate) > 50:
                summary = candidate
                break

    cot_log["summary_chars"] = len(summary)
    return summary, cot_log


# =========================================================
# ORIGINAL KG TRIPLE EXTRACTION (for KG-Diff, reuses global nlp)
# =========================================================

def extract_triples_as_tuples(sentences):
    """Extract (subject, verb, object) triples using global nlp. No reload."""
    if not sentences: return []
    # Use batch extraction via svo_per_sent
    svo_per_sent = extract_svo_triples_from_doc(sentences)
    return [triple for triples in svo_per_sent for triple in triples]

def triple_to_text(t): return f"{t[0]} {t[1]} {t[2]}"


# =========================================================
# SEMANTIC KG COVERAGE (Novel Idea 7) — unchanged
# =========================================================

def compute_semantic_kg_coverage(critical_triples, summary_triples,
                                  threshold=KG_SEMANTIC_THRESHOLD):
    if not critical_triples: return 1.0, [], []
    if not summary_triples:
        unc = [(t, 0.0) for t in critical_triples]
        return 0.0, [(t, 0.0, False) for t in critical_triples], unc
    c_embs = embed_with_legalbert([triple_to_text(t) for t in critical_triples])
    s_embs = embed_with_legalbert([triple_to_text(t) for t in summary_triples])
    sim    = cosine_similarity(c_embs, s_embs)
    per, unc, covered = [], [], 0
    for i, ct in enumerate(critical_triples):
        ms = float(sim[i].max()); ic = ms >= threshold
        per.append((ct, ms, ic))
        if ic: covered += 1
        else:  unc.append((ct, ms))
    unc.sort(key=lambda x: x[1])
    return covered / len(critical_triples), per, unc

def get_missing_entities_from_uncovered(unc, top_k=KG_TOP_MISSING):
    missing = set()
    for (subj, rel, obj), _ in unc[:top_k]:
        for tok in subj.split() + obj.split():
            if len(tok) > 3: missing.add(tok.lower())
    return missing


# =========================================================
# LCS FUNCTIONS (Novel Idea 8) — unchanged
# =========================================================

def compute_token_lcs_length(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    n = len(b); prev = [0]*(n+1); curr = [0]*(n+1)
    for ta in a:
        for j, tb in enumerate(b):
            curr[j+1] = prev[j]+1 if ta.lower()==tb.lower() else max(curr[j], prev[j+1])
        prev, curr = curr, [0]*(n+1)
    return prev[n]

def compute_lcs_score(sentence, source_sentences):
    if not source_sentences: return 0.0
    st = word_tokenize(sentence.lower())
    if not st: return 0.0
    best = 0.0
    for src in source_sentences:
        srt = word_tokenize(src.lower())
        if not srt: continue
        l = compute_token_lcs_length(st, srt)
        best = max(best, l / max(len(st), len(srt)))
    return best

def find_source_position(sentence, doc_annotation):
    best_pos, best_score = 999, -1.0
    st = word_tokenize(sentence.lower())
    for s in doc_annotation["sentences"]:
        l = compute_token_lcs_length(st, word_tokenize(s["sentence"].lower()))
        if l > best_score: best_score = l; best_pos = s["index"]
    return best_pos

def fuse_adjacent_sentences(a, b, min_ngram=LCS_MIN_NGRAM_OVERLAP):
    ta = word_tokenize(a.lower()); tb = word_tokenize(b.lower())
    for n in range(min(15, len(ta), len(tb)), min_ngram-1, -1):
        if ta[-n:] == tb[:n]:
            wb = word_tokenize(b); rem = wb[n:]
            if rem:
                rem[0] = rem[0].lower()
                return f"{a.rstrip('. ')}, {' '.join(rem)}"
    return f"{a} {b}"

def inject_connectives(sentences, connectives=LCS_CONNECTIVES):
    if not sentences: return sentences
    triggers = {"it","this","they","the same","such","these","those"}
    result = [sentences[0]]; ci = 0
    for sent in sentences[1:]:
        fw = sent.split()[0].lower() if sent.split() else ""
        if fw in triggers:
            conn = connectives[ci % len(connectives)]; ci += 1
            result.append(f"{conn} {sent[0].lower()}{sent[1:]}"
                          if conn.endswith("that")
                          else f"{conn} {sent}")
        else:
            result.append(sent)
    return result

def lcs_guided_sentence_fusion(summary, doc_annotation, normalized_weights):
    anchor_sents = [s["sentence"] for s in doc_annotation["sentences"]
                    if s["role"] in LCS_ANCHOR_ROLES]
    if not anchor_sents:
        anchor_sents = [s["sentence"] for s in doc_annotation["sentences"]]
    sents = sent_tokenize(summary)
    if not sents: return summary, {}

    s_embs = embed_with_legalbert(sents)
    a_embs = embed_with_legalbert(anchor_sents)
    sem    = cosine_similarity(s_embs, a_embs).max(axis=1)

    scored = [{"sentence": s,
               "anchor_score": LCS_ANCHOR_LCS_WEIGHT * compute_lcs_score(s, anchor_sents)
                               + LCS_ANCHOR_SEMANTIC_WEIGHT * float(sem[i])}
              for i, s in enumerate(sents)]

    selected = sorted(scored, key=lambda x: x["anchor_score"], reverse=True)[:LCS_MAX_OUTPUT_SENTENCES]
    for it in selected: it["source_pos"] = find_source_position(it["sentence"], doc_annotation)
    ordered  = [it["sentence"] for it in sorted(selected, key=lambda x: x["source_pos"])]

    fused = [ordered[0]]
    for s in ordered[1:]:
        m = fuse_adjacent_sentences(fused[-1], s, LCS_MIN_NGRAM_OVERLAP)
        if m != f"{fused[-1]} {s}": fused[-1] = m
        else: fused.append(s)

    return " ".join(inject_connectives(fused, LCS_CONNECTIVES)), {}


# =========================================================
# KG-DIFF ITERATIVE REFINEMENT (Novel Ideas 5+7) — uses global nlp
# =========================================================

def kg_iterative_refinement(summaries_dict, doc_annotation,
                             max_iterations=KG_MAX_ITERATIONS):
    critical_sents = [s["sentence"] for s in doc_annotation["sentences"]
                      if s["role"] in KG_CRITICAL_ROLES]
    if not critical_sents:
        return summaries_dict.get("LED", ""), {}

    critical_triples = extract_triples_as_tuples(critical_sents)
    if not critical_triples:
        return summaries_dict.get("LED", ""), {}

    current = summaries_dict.get("LED", "") or summaries_dict.get("PEGASUS", "")

    for _ in range(max_iterations):
        s_triples = extract_triples_as_tuples(sent_tokenize(current))
        coverage, _, uncovered = compute_semantic_kg_coverage(
            critical_triples, s_triples, KG_SEMANTIC_THRESHOLD)
        if coverage >= KG_COVERAGE_THRESHOLD: break
        if not uncovered: break

        missing  = get_missing_entities_from_uncovered(uncovered, KG_TOP_MISSING)
        corr     = []
        for s in doc_annotation["sentences"]:
            sl = s["sentence"].lower()
            if any(e in sl for e in missing):
                corr.append((0 if s["role"] in KG_CRITICAL_ROLES else 1, s["sentence"]))
        corr.sort(key=lambda x: x[0])
        corr = [s for _, s in corr[:KG_MAX_CORRECTION_SENTS]]
        if not corr: break

        inp = (f"Improve this legal summary by including the missing information.\n\n"
               f"Current summary: {current}\n\nMissing information: {' '.join(corr)}"
               f"\n\nImproved summary:")
        refined  = generate_summary("PEGASUS", inp)
        ref_trip = extract_triples_as_tuples(sent_tokenize(refined))
        ref_cov, _, _ = compute_semantic_kg_coverage(
            critical_triples, ref_trip, KG_SEMANTIC_THRESHOLD)
        if ref_cov > coverage: current = refined

    return current, {}


# =========================================================
# ANNOTATION, CONTRIBUTION, NORMALIZATION — unchanged
# =========================================================

def create_role_annotations(data, output_file):
    print("\n📋 STEP 1: CREATING SENTENCE-ROLE ANNOTATIONS")
    annotations = []
    for idx, item in enumerate(tqdm(data, desc="Annotating")):
        sentences = sent_tokenize(item.get("judgment", ""))
        if not sentences: continue
        roles_8   = classify_roles(sentences)
        ann = {"doc_id": item.get("id", idx),
               "total_sentences": len(sentences),
               "sentences": [{"index": i, "sentence": s, "role": r}
                              for i, (s, r) in enumerate(zip(sentences, roles_8))]}
        annotations.append(ann)
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump(annotations, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved {len(annotations)} annotations → {output_file}")
    return annotations


def calculate_role_contribution(train_annotations, train_data, output_file):
    print("\n📋 STEP 2: CALCULATING ROLE CONTRIBUTION SCORES")
    role_total = Counter(); role_sum = Counter()
    for ann, item in tqdm(zip(train_annotations, train_data), total=len(train_annotations)):
        ref_sents  = sent_tokenize(item.get("reference_summary", ""))
        doc_sents  = [s["sentence"] for s in ann["sentences"]]
        doc_roles  = [s["role"]     for s in ann["sentences"]]
        for r in doc_roles: role_total[r] += 1
        if doc_sents and ref_sents:
            d_embs = embed_with_legalbert(doc_sents)
            r_embs = embed_with_legalbert(ref_sents)
            for re in r_embs:
                sims = cosine_similarity([re], d_embs)[0]
                bi   = np.argmax(sims)
                if sims[bi] > 0.7: role_sum[doc_roles[bi]] += 1
    scores = {r: role_sum[r]/role_total[r] if role_total[r]>0 else 0.0
              for r in LABELS_8}
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump({"role_scores": scores, "role_total_counts": dict(role_total),
                   "role_summary_counts": dict(role_sum)}, f, indent=2, ensure_ascii=False)
    print(f"✅ Saved contribution scores → {output_file}")
    return scores


def normalize_role_weights(role_scores, output_file):
    print("\n📋 STEP 3: NORMALIZING ROLE WEIGHTS")
    total = sum(role_scores.values())
    nw    = {r: s/total if total > 0 else 1/len(LABELS_8)
             for r, s in role_scores.items()}
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump({"normalized_weights": nw, "original_scores": role_scores},
                  f, indent=2, ensure_ascii=False)
    print(f"✅ Saved weights → {output_file}")
    return nw


def hybrid_role_salience_selection(doc_annotation, normalized_weights, target_sentences):
    sentences = [s["sentence"] for s in doc_annotation["sentences"]]
    roles     = [s["role"]     for s in doc_annotation["sentences"]]
    if not sentences: return "", {}
    embeddings   = embed_with_legalbert(sentences)
    centroid     = embeddings.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    scores = [(HYBRID_ALPHA * normalized_weights.get(r, 0) * 10 + HYBRID_BETA * float(s), i)
              for i, (r, s) in enumerate(zip(roles, similarities))]
    selected = sorted([i for _, i in sorted(scores, reverse=True)[:target_sentences]])
    return " ".join(sentences[i] for i in selected), {"selected": len(selected)}


# =========================================================
# LOAD SUMMARIZATION MODELS
# =========================================================

print("\n📚 Loading summarization models...")
models = {}; tokenizers = {}

for mn, Cls, path in [
    ("BART",    AutoModelForSeq2SeqLM,          MODEL_PATHS["BART"]),
    ("LED",     LEDForConditionalGeneration,     MODEL_PATHS["LED"]),
    ("PEGASUS", PegasusForConditionalGeneration, MODEL_PATHS["PEGASUS"])
]:
    print(f"  Loading {mn}...")
    tokenizers[mn] = AutoTokenizer.from_pretrained(path)
    models[mn]     = Cls.from_pretrained(path).to(device)
    models[mn].eval()
    print(f"  ✅ {mn} loaded")

print("✅ All models loaded!")


def generate_summary(model_name, filtered_text):
    """Standard generation — no CoT."""
    model  = models[model_name]; tok = tokenizers[model_name]
    config = MODEL_CONFIG[model_name]
    inputs = tok(filtered_text, return_tensors="pt",
                 truncation=True, max_length=config["max_input"]).to(device)
    with torch.no_grad():
        if model_name == "LED":
            gam = torch.zeros_like(inputs["attention_mask"]); gam[:, 0] = 1
            ids = model.generate(inputs["input_ids"],
                                  attention_mask=inputs["attention_mask"],
                                  global_attention_mask=gam,
                                  max_length=config["max_output"],
                                  num_beams=config["num_beams"],
                                  early_stopping=True, no_repeat_ngram_size=3)
        else:
            ids = model.generate(inputs["input_ids"],
                                  attention_mask=inputs["attention_mask"],
                                  max_length=config["max_output"],
                                  num_beams=config["num_beams"],
                                  early_stopping=True, no_repeat_ngram_size=3)
    return tok.decode(ids[0], skip_special_tokens=True)


# ── Ensemble helpers ──────────────────────────────────────

def ensemble_voting(sd):
    all_s = []; [all_s.extend(sent_tokenize(s)) for s in sd.values()]
    c = Counter(s.lower().strip() for s in all_s)
    sel = [s for s, n in c.items() if n >= 2]
    return " ".join(sel or [s for s, _ in c.most_common(10)])

def ensemble_weighted_concat(sd, w):
    parts = []
    for mn, s in sd.items():
        sents = sent_tokenize(s)
        parts.extend(sents[:max(1, int(len(sents)*w[mn]))])
    seen = set(); u = []
    for s in parts:
        n = s.lower().strip()
        if n not in seen: seen.add(n); u.append(s)
    return " ".join(u)

def ensemble_best_model(sd, ref):
    best, bs = "", -1
    for mn, s in sd.items():
        sc = rouge_metric.compute(predictions=[s], references=[ref],
                                   use_stemmer=True)["rouge2"]
        if sc > bs: bs = sc; best = s
    return best

def ensemble_sentence_ranking(sd):
    pos = {}
    for mn, s in sd.items():
        for i, sent in enumerate(sent_tokenize(s)):
            n = sent.lower().strip()
            if n not in pos: pos[n] = []
            pos[n].append(i)
    ranked = sorted(pos.items(), key=lambda x: np.mean(x[1]))
    return " ".join(s for s, _ in ranked[:15])


# =========================================================
# EVALUATION
# =========================================================

print("\n📊 Loading evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_metrics(predictions, references):
    rs = rouge_metric.compute(predictions=predictions, references=references,
                               use_stemmer=True)
    bs = bertscore_metric.compute(predictions=predictions, references=references,
                                   model_type="roberta-large", lang="en",
                                   device=device, batch_size=8)
    return {"rouge1": float(rs["rouge1"]), "rouge2": float(rs["rouge2"]),
            "rougeL": float(rs["rougeL"]),
            "bertscore_f1": float(np.mean(bs["f1"]))}


# =========================================================
# MAIN PIPELINE
# =========================================================

def main():
    print("\n" + "="*70)
    print("🚀 ENSEMBLE SUMMARIZATION — FULL KG + CoT PIPELINE")
    print("   Novel Idea 5+7: KG-Diff + Semantic Coverage")
    print("   Novel Idea 8:   LCS-Guided Sentence Fusion")
    print("   🆕 Full KG:     Typed role-aware knowledge graph")
    print("   🆕 CoT:         Chain-of-Thought structured prompting")
    print("="*70)

    # ── Load data ──────────────────────────────────────────────────────────
    with open(TRAIN_JSON_PATH, 'r', encoding='utf8') as f:
        train_data = json.load(f)[:MAX_TRAIN_SAMPLES]
    with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
        test_data  = json.load(f)
    print(f"📂 {len(train_data)} train, {len(test_data)} test samples loaded")

    # ── Steps 1-3: Annotations + Weights ──────────────────────────────────
    role_path = os.path.join(OUTPUT_DIR, ROLE_CLASSIFICATION_FILE)
    train_annotations = (json.load(open(role_path, 'r', encoding='utf8'))
                         if os.path.exists(role_path)
                         else create_role_annotations(train_data, role_path))

    contrib_path = os.path.join(OUTPUT_DIR, ROLE_CONTRIBUTION_FILE)
    role_scores  = (json.load(open(contrib_path, 'r', encoding='utf8'))["role_scores"]
                    if os.path.exists(contrib_path)
                    else calculate_role_contribution(train_annotations, train_data, contrib_path))

    weights_path       = os.path.join(OUTPUT_DIR, ROLE_WEIGHTS_FILE)
    normalized_weights = (json.load(open(weights_path, 'r', encoding='utf8'))["normalized_weights"]
                          if os.path.exists(weights_path)
                          else normalize_role_weights(role_scores, weights_path))

    test_ann_path  = os.path.join(OUTPUT_DIR, "test_" + ROLE_CLASSIFICATION_FILE)
    test_annotations = (json.load(open(test_ann_path, 'r', encoding='utf8'))
                        if os.path.exists(test_ann_path)
                        else create_role_annotations(test_data, test_ann_path))

    # ── STEP 5: Generate summaries ─────────────────────────────────────────
    print("\n" + "="*70)
    print("STEP 5: GENERATING SUMMARIES — FULL KG + CoT")
    print("="*70)

    all_model_summaries = {"BART": [], "LED": [], "PEGASUS": []}

    ensemble_summaries = {
        "voting":        [],
        "weighted":      [],
        "best_model":    [],
        "ranking":       [],
        "kg_refined":    [],
        "lcs_fused":     [],
        "cot_led":       [],   # 🆕 CoT with LED (full context)
        "cot_pegasus":   [],   # 🆕 CoT with PEGASUS
        "kg_cot_fused":  []    # 🆕 Full pipeline: KG → CoT → LCS Fusion
    }

    references      = []
    all_kg_logs     = []
    all_cot_logs    = []

    print("\n🔄 Processing test documents...")
    for test_annotation, test_item in tqdm(
        zip(test_annotations, test_data), total=len(test_data), desc="Processing"
    ):
        reference = test_item["reference_summary"]
        references.append(reference)

        doc_length       = test_annotation["total_sentences"]
        adaptive_targets = get_adaptive_targets(doc_length)
        summaries_dict   = {}

        # ── Standard model summaries ───────────────────────────────────────
        for mn in ["BART", "LED", "PEGASUS"]:
            ft, _ = hybrid_role_salience_selection(
                test_annotation, normalized_weights, adaptive_targets[mn])
            s = generate_summary(mn, ft)
            all_model_summaries[mn].append(s)
            summaries_dict[mn] = s

        # ── Standard ensemble strategies ──────────────────────────────────
        ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
        ensemble_summaries["weighted"].append(
            ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
        ensemble_summaries["best_model"].append(
            ensemble_best_model(summaries_dict, reference))
        ensemble_summaries["ranking"].append(
            ensemble_sentence_ranking(summaries_dict))

        # ── KG-Diff (Novel 5+7) ────────────────────────────────────────────
        kg_refined, _ = kg_iterative_refinement(summaries_dict, test_annotation)
        ensemble_summaries["kg_refined"].append(kg_refined)

        # ── LCS Fusion (Novel 8) ───────────────────────────────────────────
        lcs_fused, _ = lcs_guided_sentence_fusion(
            kg_refined, test_annotation, normalized_weights)
        ensemble_summaries["lcs_fused"].append(lcs_fused)

        # ── 🆕 FULL KG CONSTRUCTION ─────────────────────────────────────────
        # Build typed legal KG using en_legal_ner_trf (already loaded globally)
        # TEXT CHANGE ONLY: the only thing that changes per document is the
        # doc_annotation (sentences + roles) — the nlp model itself is not reloaded
        kg, kg_log = build_full_legal_kg(test_annotation, normalized_weights)
        if len(all_kg_logs) < 3:
            all_kg_logs.append({"doc_id": test_annotation["doc_id"], "log": kg_log})

        # ── 🆕 CoT SUMMARIZATION (LED — full context) ──────────────────────
        led_target = adaptive_targets["LED"]
        led_ft, _  = hybrid_role_salience_selection(
            test_annotation, normalized_weights, led_target)

        cot_led, cot_log_led = generate_cot_summary(
            "LED", led_ft, test_annotation, kg, normalized_weights)
        ensemble_summaries["cot_led"].append(cot_led)

        if len(all_cot_logs) < 3:
            all_cot_logs.append({"doc_id": test_annotation["doc_id"],
                                  "log": cot_log_led})

        # ── 🆕 CoT SUMMARIZATION (PEGASUS) ────────────────────────────────
        peg_target = adaptive_targets["PEGASUS"]
        peg_ft, _  = hybrid_role_salience_selection(
            test_annotation, normalized_weights, peg_target)

        cot_peg, _ = generate_cot_summary(
            "PEGASUS", peg_ft, test_annotation, kg, normalized_weights)
        ensemble_summaries["cot_pegasus"].append(cot_peg)

        # ── 🆕 FULL PIPELINE: KG → CoT (LED) → LCS Fusion ─────────────────
        # Best expected pipeline:
        # 1. Full KG provides grounded issue/reasoning/decision context
        # 2. CoT prompts LED with that structured context → ordered summary
        # 3. LCS Fusion refines the CoT output for maximum ROUGE-L
        kg_cot_fused, _ = lcs_guided_sentence_fusion(
            cot_led, test_annotation, normalized_weights)
        ensemble_summaries["kg_cot_fused"].append(kg_cot_fused)

    print("✅ All summaries generated!")

    # ── Save logs ──────────────────────────────────────────────────────────
    for fname, data in [
        ("full_kg_construction_logs.json", all_kg_logs),
        ("cot_prompting_logs.json",        all_cot_logs)
    ]:
        with open(os.path.join(OUTPUT_DIR, fname), 'w', encoding='utf8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"📊 Logs saved to {OUTPUT_DIR}/")

    # ── Evaluate ──────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("📊 EVALUATING ALL APPROACHES")
    print("="*70)

    results = {}
    for mn in ["BART", "LED", "PEGASUS"]:
        m = compute_metrics(all_model_summaries[mn], references)
        results[mn] = m
        print(f"  {mn}: R1={m['rouge1']:.4f} R2={m['rouge2']:.4f} "
              f"RL={m['rougeL']:.4f} BS={m['bertscore_f1']:.4f}")

    labels = {
        "voting":       "Ensemble-Voting",
        "weighted":     "Ensemble-Weighted",
        "best_model":   "Ensemble-BestModel",
        "ranking":      "Ensemble-Ranking",
        "kg_refined":   "KG-Refined (5+7)",
        "lcs_fused":    "LCS-Fused (8) ✨",
        "cot_led":      "🆕 CoT-LED",
        "cot_pegasus":  "🆕 CoT-PEGASUS",
        "kg_cot_fused": "🆕 FullKG+CoT+LCS (BEST)"
    }
    for strategy, label in labels.items():
        m = compute_metrics(ensemble_summaries[strategy], references)
        results[f"ensemble_{strategy}"] = m
        tag = " 🎯 TARGET MET!" if m["rougeL"] >= 0.34 else ""
        print(f"  {label:<35} R1={m['rouge1']:.4f} R2={m['rouge2']:.4f} "
              f"RL={m['rougeL']:.4f}{tag} BS={m['bertscore_f1']:.4f}")

    # ── Final comparison table ─────────────────────────────────────────────
    print("\n" + "="*80)
    print("📊 FINAL RESULTS — FULL KG + CoT vs BASELINES")
    print("="*80)
    sorted_r = sorted(results.items(), key=lambda x: x[1]["rougeL"], reverse=True)
    print(f"{'Approach':<40} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<12} {'BERTScore'}")
    print("-"*80)
    for app, m in sorted_r:
        tgt = " ✓" if m["rougeL"] >= 0.34 else f" ({0.34-m['rougeL']:.4f}↓)"
        print(f"{app:<40} {m['rouge1']:.4f}     {m['rouge2']:.4f}     "
              f"{m['rougeL']:.4f}{tgt:<10} {m['bertscore_f1']:.4f}")

    # ── Impact analysis ────────────────────────────────────────────────────
    print("\n" + "="*80)
    print("🔬 FULL KG + CoT IMPACT ANALYSIS")
    print("="*80)
    baseline = results.get("LED", {}).get("rougeL", 0.0)
    print(f"  Baseline LED ROUGE-L: {baseline:.4f}\n")
    for key, label in [
        ("ensemble_cot_led",      "🆕 CoT-LED"),
        ("ensemble_cot_pegasus",  "🆕 CoT-PEGASUS"),
        ("ensemble_kg_cot_fused", "🆕 FullKG+CoT+LCS (Full Pipeline)"),
        ("ensemble_lcs_fused",    "LCS-Fused (Novel 8 baseline)")
    ]:
        rl    = results.get(key, {}).get("rougeL", 0.0)
        delta = rl - baseline
        sign  = "+" if delta >= 0 else ""
        print(f"  {label:<40} RL={rl:.4f}  Δ={sign}{delta:.4f}")

    # ── Save results ───────────────────────────────────────────────────────
    complete = {
        "experiment": "Full KG + CoT-Guided Summarization",
        "novel_contributions": {
            "full_kg": {
                "spacy_model":   "en_legal_ner_trf (loaded ONCE globally)",
                "node_types":    list(ROLE_TO_NODE_TYPE.values()),
                "edge_types":    ["ENTITY_CO_OCCURRENCE", "GIVES_RISE_TO",
                                  "ADDRESSED_BY", "RESOLVED_BY", "SUPPORTS",
                                  "TEMPORAL_PRECEDES", "SVO_*"],
                "optimization":  "nlp.pipe() batch processing — no per-sentence reload"
            },
            "cot_prompting": {
                "structure":     "issue → reasoning → holding",
                "why_rouge_l":   "Matches reference abstract structure, maximises LCS",
                "models_used":   ["LED (full context)", "PEGASUS (short context)"],
                "grounding":     "KG-extracted entities + salient paths"
            }
        },
        "results": results
    }
    rpath = os.path.join(OUTPUT_DIR, "complete_results_fullkg_cot.json")
    with open(rpath, 'w', encoding='utf8') as f:
        json.dump(complete, f, indent=2, ensure_ascii=False)

    # Save all summaries
    for mn in ["BART", "LED", "PEGASUS"]:
        with open(os.path.join(OUTPUT_DIR, f"{mn.lower()}_summaries.json"),
                  'w', encoding='utf8') as f:
            json.dump([{"id": item.get("id", i), "generated": s, "reference": r}
                       for i, (item, s, r) in enumerate(
                           zip(test_data, all_model_summaries[mn], references))],
                      f, indent=2, ensure_ascii=False)

    for strategy in ensemble_summaries:
        with open(os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json"),
                  'w', encoding='utf8') as f:
            json.dump([{"id": item.get("id", i), "generated": s, "reference": r}
                       for i, (item, s, r) in enumerate(
                           zip(test_data, ensemble_summaries[strategy], references))],
                      f, indent=2, ensure_ascii=False)

    print(f"\n✅ Results saved → {rpath}")
    print("\n" + "="*70)
    print("✅ FULL KG + CoT PIPELINE COMPLETE!")
    print("="*70)
    print("\n🆕 Key optimizations:")
    print("   spaCy (en_legal_ner_trf): loaded ONCE at startup")
    print("   Entity extraction:        nlp.pipe() batch — no per-doc reload")
    print("   SVO extraction:           nlp.pipe() batch — one pass per document")
    print("   CoT prompts:              grounded with KG entities + salient paths")
    print("   Full pipeline:            KG construction → CoT generation → LCS fusion")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted")
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()

📥 Downloading NLTK data...
🚀 Device: cuda

📚 Loading spaCy legal NER model (en_legal_ner_trf)...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_legal_ner_trf' (3.2.0) was trained with spaCy v3.2.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'transformers' and 'spacy-transformers' versions. For more details and available updates, run: python -m spacy validate
  war

✅ en_legal_ner_trf loaded successfully

📋 CONFIGURATION — FULL KG + CoT SUMMARIZATION
  🆕 Full KG Construction with en_legal_ner_trf (loaded once)
  🆕 Chain-of-Thought (CoT) Guided Summarization
  ✨ LCS-Guided Sentence Fusion (ROUGE-L booster)
  Output: ./ensemble_results_fullkg_cot

📚 Loading HSLN role classifier...
✅ HSLN loaded

📚 Loading InLegalBERT...
✅ InLegalBERT loaded

📚 Loading summarization models...
  Loading BART...
  ✅ BART loaded
  Loading LED...
  ✅ LED loaded
  Loading PEGASUS...
  ✅ PEGASUS loaded
✅ All models loaded!

📊 Loading evaluation metrics...

🚀 ENSEMBLE SUMMARIZATION — FULL KG + CoT PIPELINE
   Novel Idea 5+7: KG-Diff + Semantic Coverage
   Novel Idea 8:   LCS-Guided Sentence Fusion
   🆕 Full KG:     Typed role-aware knowledge graph
   🆕 CoT:         Chain-of-Thought structured prompting
📂 3000 train, 100 test samples loaded

📋 STEP 1: CREATING SENTENCE-ROLE ANNOTATIONS


Annotating: 100%|███████████████████████████████████████████████████████████████████| 3000/3000 [19:23<00:00,  2.58it/s]


✅ Saved 3000 annotations → ./ensemble_results_fullkg_cot/sentence_role_annotations_8label.json

📋 STEP 2: CALCULATING ROLE CONTRIBUTION SCORES


...

✅ Saved 100 annotations → ./ensemble_results_fullkg_cot/test_sentence_role_annotations_8label.json

STEP 5: GENERATING SUMMARIES — FULL KG + CoT

🔄 Processing test documents...


Processing: 100%|██████████████████████████████████████████████████████████████████| 100/100 [3:19:56<00:00, 119.97s/it]


✅ All summaries generated!
📊 Logs saved to ./ensemble_results_fullkg_cot/

📊 EVALUATING ALL APPROACHES


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  BART: R1=0.3702 R2=0.1867 RL=0.2100 BS=0.8497
  LED: R1=0.5011 R2=0.2652 RL=0.2754 BS=0.8529
  PEGASUS: R1=0.3822 R2=0.1894 RL=0.2141 BS=0.8431
  Ensemble-Voting                     R1=0.3368 R2=0.1646 RL=0.1760 BS=0.8370
  Ensemble-Weighted                   R1=0.4027 R2=0.1997 RL=0.2245 BS=0.8461
  Ensemble-BestModel                  R1=0.5018 R2=0.2811 RL=0.2838 BS=0.8578
  Ensemble-Ranking                    R1=0.4932 R2=0.2412 RL=0.2372 BS=0.8374
  KG-Refined (5+7)                    R1=0.5011 R2=0.2652 RL=0.2754 BS=0.8529
  LCS-Fused (8) ✨                     R1=0.5021 R2=0.2646 RL=0.2737 BS=0.8523
  🆕 CoT-LED                           R1=0.3866 R2=0.1805 RL=0.2095 BS=0.8397
  🆕 CoT-PEGASUS                       R1=0.1968 R2=0.0863 RL=0.1193 BS=0.8197
  🆕 FullKG+CoT+LCS (BEST)             R1=0.3882 R2=0.1811 RL=0.2076 BS=0.8392

📊 FINAL RESULTS — FULL KG + CoT vs BASELINES
Approach                                 ROUGE-1    ROUGE-2    ROUGE-L      BERTScore
--------------------